In [0]:
#Inicia o df_bi em um novo notebook, sem nenhuma perda do notebook anterior, com a sessão spark ativa
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

df_bi = spark.read.table("main.default.df_bi")

df_bi.printSchema()

In [0]:
from pyspark.sql import functions as F

df_bi = (
    df_bi

    # ── 1. PLACAR E RESULTADO
    .withColumn("placar",
        F.concat(F.col("gols_mandante"), F.lit(" x "), F.col("gols_visitante")))

    .withColumn("saldo_gols",
        F.col("gols_mandante") - F.col("gols_visitante"))

    .withColumn("gols_2_tempo_mandante",
        F.col("gols_mandante") - F.col("gols_1_tempo_mandante"))

    .withColumn("gols_2_tempo_visitante",
        F.col("gols_visitante") - F.col("gols_1_tempo_visitante"))

    .withColumn("total_gols_partida",
        F.col("gols_mandante") + F.col("gols_visitante"))

    # ── 2. EFICIÊNCIA OFENSIVA 
    .withColumn("eficiencia_finalizacao_mandante",
        F.round(F.col("gols_mandante") / F.when(F.col("chutes_mandante") == 0, None)
                .otherwise(F.col("chutes_mandante")), 2))

    .withColumn("eficiencia_finalizacao_visitante",
        F.round(F.col("gols_visitante") / F.when(F.col("chutes_visitante") == 0, None)
                .otherwise(F.col("chutes_visitante")), 2))

    .withColumn("chutes_no_alvo_mandante",
        F.col("chutes_mandante") - F.col("chutes_fora_mandante"))

    .withColumn("chutes_no_alvo_visitante",
        F.col("chutes_visitante") - F.col("chutes_fora_visitante"))

    .withColumn("precisao_chutes_mandante",
        F.round((F.col("chutes_mandante") - F.col("chutes_fora_mandante")) /
                F.when(F.col("chutes_mandante") == 0, None)
                .otherwise(F.col("chutes_mandante")) * 100, 1))

    .withColumn("precisao_chutes_visitante",
        F.round((F.col("chutes_visitante") - F.col("chutes_fora_visitante")) /
                F.when(F.col("chutes_visitante") == 0, None)
                .otherwise(F.col("chutes_visitante")) * 100, 1))

    # ── 3. CONTEXTO TEMPORAL 
    .withColumn("mes",        F.month("data"))
    .withColumn("dia_semana", F.date_format("data", "EEEE"))
    .withColumn("fase_campeonato",
        F.when(F.col("rodada") <= 10, "Inicio")
         .when(F.col("rodada") <= 25, "Meio")
         .otherwise("Final"))

    # ── 4. CONTEXTO DO JOGO 
    .withColumn("diferenca_colocacao",
        F.abs(F.col("colocacao_mandante") - F.col("colocacao_visitante")))

    .withColumn("jogo_equilibrado",
        F.when(F.abs(F.col("colocacao_mandante") - F.col("colocacao_visitante")) <= 5,
               "Equilibrado")
         .otherwise("Desequilibrado"))

    .withColumn("vantagem_valor_equipe",
        F.col("valor_equipe_titular_mandante") - F.col("valor_equipe_titular_visitante"))

    .withColumn("faixa_publico",
        F.when(F.col("publico") < 10000, "Baixo")
         .when(F.col("publico") < 30000, "Médio")
         .otherwise("Alto"))

    .withColumn("diferenca_idade_media",
        F.round(F.col("idade_media_titular_mandante") - F.col("idade_media_titular_visitante"), 1))
)



df_bi.select("time_mandante", "publico", "media_publico_time")
df_bi.printSchema()
df_bi.display()

In [0]:
from pyspark.sql import functions as F

df_bi = spark.read.table("main.default.df_bi")

# calcula agregações de público
media_publico_por_time = df_bi.groupBy("time_mandante").agg(
    F.round(F.avg("publico"), 0).cast("long").alias("media_publico_time"),
    F.max("publico").alias("maior_publico_time"),
    F.min("publico").alias("menor_publico_time"),
)

# join seguro 
df_bi = df_bi.join(media_publico_por_time, on="time_mandante", how="left")

df_bi.select("time_mandante", "publico", "media_publico_time",
             "maior_publico_time", "menor_publico_time").show(10)

df_bi.limit(10).display()

In [0]:
from pyspark.sql import functions as F

# ── RECARREGA E RECRIA AS COLUNAS DERIVADAS
df_bi = spark.read.table("main.default.brasileirao_2023_bi")

df_bi = df_bi \
    .withColumn("mes",           F.month("data")) \
    .withColumn("dia_semana",    F.date_format("data", "EEEE")) \
    .withColumn("fase_campeonato",
        F.when(F.col("rodada") <= 10, "Inicio")
         .when(F.col("rodada") <= 25, "Meio")
         .otherwise("Final"))

# ── DIM_TIME 
dim_time = df_bi.select(
    F.col("time_mandante").alias("nome_time"),
    F.col("valor_equipe_titular_mandante").alias("valor_equipe"),
    F.col("colocacao_mandante").alias("colocacao"),
    F.col("media_publico_time").alias("media_publico"),
    F.col("maior_publico_time").alias("maior_publico"),
    F.col("menor_publico_time").alias("menor_publico"),
    F.col("pontuacao_final_campeonato").alias("pontuacao_final"),
).unionByName(
    df_bi.select(
        F.col("time_visitante").alias("nome_time"),
        F.col("valor_equipe_titular_visitante").alias("valor_equipe"),
        F.col("colocacao_visitante").alias("colocacao"),
        F.lit(None).cast("long").alias("media_publico"),
        F.lit(None).cast("long").alias("maior_publico"),
        F.lit(None).cast("long").alias("menor_publico"),
        F.lit(None).cast("long").alias("pontuacao_final"),
    )
).dropDuplicates(["nome_time"]) \
 .withColumn("sk_time", F.monotonically_increasing_id())

dim_time.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("main.default.dim_time")
print("dim_time OK —", dim_time.count(), "times")

# ── DIM_DATA 
dim_data = df_bi.select("data", "rodada", "mes", "dia_semana", "fase_campeonato") \
    .dropDuplicates(["data"]) \
    .withColumn("sk_data", F.monotonically_increasing_id())

dim_data.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("main.default.dim_data")
print("dim_data OK —", dim_data.count(), "datas")

# ── DIM_ESTADIO
dim_estadio = df_bi.select(
    F.col("estadio").alias("nome_estadio")
).dropDuplicates(["nome_estadio"]) \
 .withColumn("sk_estadio", F.monotonically_increasing_id())

dim_estadio.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("main.default.dim_estadio")
print("dim_estadio OK —", dim_estadio.count(), "estádios")

# ── DIM_ARBITRO 
dim_arbitro = df_bi.select(
    F.col("arbitro").alias("nome_arbitro")
).dropDuplicates(["nome_arbitro"]) \
 .withColumn("sk_arbitro", F.monotonically_increasing_id())

dim_arbitro.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("main.default.dim_arbitro")
print("dim_arbitro OK —", dim_arbitro.count(), "árbitros")

# ── DIM_TECNICO ───────────────────────────────────────────────────────────
dim_tecnico = df_bi.select(
    F.col("tecnico_mandante").alias("nome_tecnico")
).unionByName(
    df_bi.select(F.col("tecnico_visitante").alias("nome_tecnico"))
).dropDuplicates(["nome_tecnico"]) \
 .withColumn("sk_tecnico", F.monotonically_increasing_id())

dim_tecnico.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("main.default.dim_tecnico")
print("dim_tecnico OK —", dim_tecnico.count(), "técnicos")

# ── FATO_PARTIDA ──────────────────────────────────────────────────────────
fato = df_bi \
    .join(dim_time.select("sk_time", F.col("nome_time").alias("time_mandante")),
          on="time_mandante", how="left") \
    .withColumnRenamed("sk_time", "sk_time_mandante") \
    .join(dim_time.select("sk_time", F.col("nome_time").alias("time_visitante")),
          on="time_visitante", how="left") \
    .withColumnRenamed("sk_time", "sk_time_visitante") \
    .join(dim_data.select("sk_data", "data"),
          on="data", how="left") \
    .join(dim_estadio.select("sk_estadio", F.col("nome_estadio").alias("estadio")),
          on="estadio", how="left") \
    .join(dim_arbitro.select("sk_arbitro", F.col("nome_arbitro").alias("arbitro")),
          on="arbitro", how="left") \
    .join(dim_tecnico.select("sk_tecnico", F.col("nome_tecnico").alias("tecnico_mandante")),
          on="tecnico_mandante", how="left") \
    .withColumnRenamed("sk_tecnico", "sk_tecnico_mandante") \
    .drop("time_mandante", "time_visitante", "data", "estadio",
          "arbitro", "tecnico_mandante", "tecnico_visitante",
          "colocacao_mandante", "colocacao_visitante",
          "valor_equipe_titular_mandante", "valor_equipe_titular_visitante",
          "media_publico_time", "maior_publico_time", "menor_publico_time",
          "pontuacao_final_campeonato", "mes", "dia_semana",
          "rodada", "fase_campeonato")

fato.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("main.default.fato_partida")
print("fato_partida OK —", fato.count(), "partidas")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# recria dim_time com SKs verdadeiramente únicas e sequenciais
dim_time = spark.read.table("main.default.dim_time") \
    .drop("sk_time") \
    .withColumn("sk_time", F.row_number().over(Window.orderBy("nome_time")))

dim_time.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("main.default.dim_time")

# verifica se tem duplicatas
dim_time.groupBy("sk_time").count().filter("count > 1").show()
print("Total times:", dim_time.count())

In [0]:
from pyspark.sql import functions as F

fato = spark.read.table("main.default.fato_partida")
dim_time = spark.read.table("main.default.dim_time")

# lado mandante
fato_mandante = fato.select(
    F.col("sk_time_mandante").alias("sk_time"),
    F.col("chutes_mandante").alias("chutes"),
    F.col("chutes_fora_mandante").alias("chutes_fora"),
    F.col("escanteios_mandante").alias("escanteios"),
    F.col("faltas_mandante").alias("faltas"),
    F.col("defesas_mandante").alias("defesas"),
    F.col("chutes_bola_parada_mandante").alias("chutes_bola_parada"),
    F.col("gols_mandante").alias("gols"),
    F.col("pontos_mandante").alias("pontos"),
    F.col("sk_data"),
    F.col("sk_estadio"),
    F.lit("Casa").alias("condicao")
)

# lado visitante
fato_visitante = fato.select(
    F.col("sk_time_visitante").alias("sk_time"),
    F.col("chutes_visitante").alias("chutes"),
    F.col("chutes_fora_visitante").alias("chutes_fora"),
    F.col("escanteios_visitante").alias("escanteios"),
    F.col("faltas_visitante").alias("faltas"),
    F.col("defesas_visitante").alias("defesas"),
    F.col("chutes_bola_parada_visitante").alias("chutes_bola_parada"),
    F.col("gols_visitante").alias("gols"),
    F.col("pontos_visitante").alias("pontos"),
    F.col("sk_data"),
    F.col("sk_estadio"),
    F.lit("Fora").alias("condicao")
)

fato_times = fato_mandante.unionByName(fato_visitante)

fato_times.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("main.default.fato_times")

print("fato_times OK —", fato_times.count(), "linhas")

In [0]:
spark.read.table("main.default.fato_times") \
    .groupBy("condicao", "sk_time") \
    .count() \
    .orderBy("sk_time") \
    .show(40)

In [0]:
from pyspark.sql import functions as F

fato = spark.read.table("main.default.fato_partida")
dim_time = spark.read.table("main.default.dim_time")

fato_mandante = fato.join(
    dim_time.select("sk_time", "nome_time"),
    fato["sk_time_mandante"] == dim_time["sk_time"],
    how="left"
).select(
    F.col("nome_time"),
    F.col("chutes_mandante").alias("chutes"),
    F.col("chutes_fora_mandante").alias("chutes_fora"),
    F.col("escanteios_mandante").alias("escanteios"),
    F.col("faltas_mandante").alias("faltas"),
    F.col("defesas_mandante").alias("defesas"),
    F.col("gols_mandante").alias("gols"),
    F.col("pontos_mandante").alias("pontos"),
    F.col("sk_data"),
    F.lit("Casa").alias("condicao")
)

fato_visitante = fato.join(
    dim_time.select("sk_time", "nome_time"),
    fato["sk_time_visitante"] == dim_time["sk_time"],
    how="left"
).select(
    F.col("nome_time"),
    F.col("chutes_visitante").alias("chutes"),
    F.col("chutes_fora_visitante").alias("chutes_fora"),
    F.col("escanteios_visitante").alias("escanteios"),
    F.col("faltas_visitante").alias("faltas"),
    F.col("defesas_visitante").alias("defesas"),
    F.col("gols_visitante").alias("gols"),
    F.col("pontos_visitante").alias("pontos"),
    F.col("sk_data"),
    F.lit("Fora").alias("condicao")
)

fato_times = fato_mandante.unionByName(fato_visitante)

fato_times.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("main.default.fato_times")

fato_times.groupBy("nome_time", "condicao") \
    .agg(F.sum("chutes").alias("total_chutes")) \
    .orderBy("nome_time") \
    .show(40)